# Document Intelligence with RAG

A system that answers questions about any PDF using Retrieval-Augmented Generation.
It understands the meaning of a question, retrieves the most relevant passages from
the document, and generates a grounded answer with Google's Gemini model. If the
answer isn't in the document, it says so rather than making one up.

**Live app:** https://huggingface.co/spaces/Sharonmary/document-intelligence-rag

## Setup

I connect to Google's Gemini model using a free-tier API key, stored securely as a
Colab secret so it never appears in the code.

In [1]:
from google.colab import userdata

# Securely fetch the key from Colab's vault (the key itself never shows)
api_key = userdata.get('RAG_PROJECT')

if api_key:
    print("API key loaded successfully. Length:", len(api_key))
else:
    print("Key not found — check the secret name matches exactly")

API key loaded successfully. Length: 53


I install the libraries: google-genai for Gemini, pypdf to read PDFs, and
sentence-transformers for the embeddings.

In [2]:
!pip install google-genai pypdf sentence-transformers -q
print("Libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 5.0 MB/s eta 0:00:00
Libraries installed.


## Testing the Gemini Connection

Before building anything, I send a simple test message to confirm the connection to
Gemini works end to end. If it replies, the API key and model are set up correctly
and I can move on to building the pipeline.

In [3]:
from google import genai
from google.colab import userdata

# Create the client with the secure key
client = genai.Client(api_key=userdata.get('RAG_PROJECT'))

# Quick test using a current model
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one short sentence."
)
print(response.text)

Hello!


## Step 1: Read the Document

I extract the raw text from a PDF. Here I use Warren Buffett's 2023 shareholder
letter as a test document, since it is public, text-rich, and the kind of financial
document an analyst would actually query.

In [4]:
import urllib.request

# Download a sample PDF (a public financial/policy-style document)
url = "https://www.berkshirehathaway.com/letters/2023ltr.pdf"
urllib.request.urlretrieve(url, "document.pdf")
print("Document downloaded as document.pdf")

Document downloaded as document.pdf


With the PDF downloaded, I extract its text. The code opens the file and loops
through every page, pulling the text into one long string. The preview confirms it
read correctly: this is the opening of Buffett's letter, his tribute to Charlie
Munger.

In [5]:
from pypdf import PdfReader

# Read the PDF and pull out all its text
reader = PdfReader("/content/document.pdf")

full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print("Total characters extracted:", len(full_text))
print("\nFirst 500 characters:\n")
print(full_text[:500])

Total characters extracted: 46301

First 500 characters:

Charlie Munger – The Architect of Berkshire Hathaway  
Charlie Munger died on November 28, just 33 days before his 100th birthday. 
Though born and raised in Omaha, he spent 80% of his life domiciled 
elsewhere. Consequently, it was not until 1959 when he was 35 that I first met him. 
In 1962, he decided that he should take up money management. 
Three years later he told me – correctly! – that I had made a dumb decision in 
buying control of Berkshire. But, he assured me, since I had already mad


## Step 2: Chunk the Text

The document is split into overlapping passages of about 800 characters each.
Chunking lets the system later retrieve only the passages relevant to a question,
instead of sending the whole document every time. The 100-character overlap stops an
idea being sliced in half at a chunk boundary.

In [6]:
def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap  # step forward, but overlap a bit
    return chunks

chunks = chunk_text(full_text)

print("Number of chunks created:", len(chunks))
print("\nExample chunk (chunk #5):\n")
print(chunks[5])

Number of chunks created: 67

Example chunk (chunk #5):

tracted an unusual number of such “lifetime” shareholders 
and their heirs. We cherish their presence and believe they are entitled to hear every year both the 
good and bad news, delivered directly from their CEO and not from an investor-relations officer 
or communications consultant forever serving up optimism and syrupy mush. 
In visualizing the owners that Berkshire seeks, I am lucky to have the perfect mental model, 
my sister, Bertie. Let me introduce her. 
For openers, Bertie is smart, wise and likes to challenge my thinking. We have never, 
however, had a shouting match or anything close to a ruptured relationship. We never will. 
Furthermore, Bertie, and her three daughters as well, have a large portion of their savings 
in Berkshire shares. Their ownership spans decades, and eve


A quick check to see the overlap in action: the last 100 characters of chunk 0 are
identical to the first 100 characters of chunk 1.

In [7]:
print("End of chunk 0:", repr(chunks[0][-100:]))
print("\nStart of chunk 1:", repr(chunks[1][:100]))

End of chunk 0: 'en managing and whose \nmoney I had used for the Berkshire purchase. Moreover, neither of us expected'

Start of chunk 1: 'en managing and whose \nmoney I had used for the Berkshire purchase. Moreover, neither of us expected'


## Step 3: Embed the Chunks

Each chunk becomes a vector of 384 numbers capturing its meaning. Chunks with similar
meaning get similar vectors, which is what lets the system later match a question to
passages by meaning rather than exact words.

In [8]:
from sentence_transformers import SentenceTransformer

# Load a small, fast, free embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert every chunk into a vector of numbers (its "meaning")
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)

print("Shape of embeddings:", chunk_embeddings.shape)
print("Each chunk is now a vector of", chunk_embeddings.shape[1], "numbers")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Shape of embeddings: (67, 384)
Each chunk is now a vector of 384 numbers


## Step 4: Retrieve Relevant Passages

When a question comes in, it is embedded the same way and compared against every
chunk to find the most similar ones. This is semantic search. Notice it matches on
meaning: a question using "die" still finds the passage that says "died".

In [9]:
import numpy as np

def retrieve(question, top_k=3):
    # Turn the question into a vector, same as we did for chunks
    q_embedding = embed_model.encode([question])

    # Measure similarity between the question and every chunk
    # (cosine similarity: higher = more similar in meaning)
    similarities = np.dot(chunk_embeddings, q_embedding.T).flatten()

    # Find the top_k most similar chunks
    top_indices = similarities.argsort()[-top_k:][::-1]

    return [chunks[i] for i in top_indices]

# Test it
question = "When did Charlie Munger die?"
results = retrieve(question)

print("Question:", question)
print("\nMost relevant chunks found:\n")
for i, chunk in enumerate(results, 1):
    print(f"--- Chunk {i} ---")
    print(chunk[:300])
    print()

Question: When did Charlie Munger die?

Most relevant chunks found:

--- Chunk 1 ---
Charlie Munger – The Architect of Berkshire Hathaway  
Charlie Munger died on November 28, just 33 days before his 100th birthday. 
Though born and raised in Omaha, he spent 80% of his life domiciled 
elsewhere. Consequently, it was not until 1959 when he was 35 that I first met him. 
In 1962, he de

--- Chunk 2 ---
en managing and whose 
money I had used for the Berkshire purchase. Moreover, neither of us expected that 
Charlie would ever own a share of Berkshire stock. 
Nevertheless, Charlie, in 1965, promptly advised me: “Warren, forget about 
ever buying another company like Berkshire. But now that you cont

--- Chunk 3 ---
dly, jerked me back to sanity when my old habits surfaced. Until his death, he 
continued in this role and together we, along with those who early on invested with 
us, ended up far better off than Charlie and I had ever dreamed possible. 
In reality, Charlie was the “architect”

## Step 5: Generate a Grounded Answer

The retrieved passages and the question go to Gemini, with a clear instruction to
answer using only the provided context. This grounding is the whole point of RAG: it
stops the model inventing answers and forces it to rely on the actual document.

In [10]:
def rag_answer(question, top_k=3):
    # Step 1-3: retrieve the most relevant chunks
    relevant_chunks = retrieve(question, top_k=top_k)
    context = "\n\n".join(relevant_chunks)

    # Step 4: build a prompt that forces the AI to use ONLY the document
    prompt = f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say "I couldn't find that in the document."

Context:
{context}

Question: {question}

Answer:"""

    # Ask Gemini
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

# Test it
question = "When did Charlie Munger die?"
answer = rag_answer(question)
print("Question:", question)
print("\nAnswer:", answer)

Question: When did Charlie Munger die?

Answer: Charlie Munger died on November 28.


## Testing the Guardrail

I test several questions, including one the document cannot answer (the capital of
France). A plain chatbot would answer "Paris" from its own knowledge, but a
trustworthy RAG system refuses, because it is grounded only in the document. This
no-hallucination behaviour is the most important property for finance and compliance.

In [13]:
# Test several questions, including one the document can't answer
test_questions = [
    "Who is Bertie?",
    "What does Buffett say about Berkshire's investments?",
    "What is the capital of France?"   # NOT in the document - tests the guardrail
]

for q in test_questions:
    print("Q:", q)
    print("A:", rag_answer(q))
    print("-" * 60)

Q: Who is Bertie?
A: Bertie is described as a woman who spent her early formative years in a middle-class neighborhood in Omaha and emerged as one of the country's great investors many decades later. She was financially active for 20 years after starting a family in 1956, holding bonds, putting 1/3 of her funds in a publicly-held mutual fund, and trading stocks. In 1980, at age 46, she decided to retain only her mutual fund and Berkshire shares, making no new trades for the next 43 years, during which time she became very rich and made large philanthropic gifts. Bertie returns to Omaha every May to be re-energized, and her investment reasoning is attributed to common sense absorbed as a child in Omaha. She understands many accounting terms, follows business news by reading four newspapers daily, but doesn't consider herself an economic expert, and is sensible, believing pundits should be ignored. The author of the text anticipates her questions and provides her with honest answers.
---

## Conclusion

This builds a complete RAG pipeline: chunk, embed, retrieve, generate, with a
guardrail against hallucination. The same logic powers the live web app, where
anyone can upload their own PDF and ask questions.

**Try the live app:** https://huggingface.co/spaces/Sharonmary/document-intelligence-rag